# PASO 3: Modelado ML - Baseline + XGBoost + SHAP
## Predicción de Deserción Estudiantil

**Objetivo:** 
1. Entrenar Regresión Logística (baseline)
2. Entrenar XGBoost con GridSearchCV (modelo principal)
3. Comparar métricas (Accuracy, F1, AUC-ROC)
4. Explicabilidad con SHAP
5. Guardar el mejor modelo

In [ ]:
# Instalación de librerías
!pip install -q xgboost shap scikit-learn matplotlib seaborn pandas numpy

In [ ]:
# Importar librerías
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score, 
    confusion_matrix, classification_report, roc_curve, auc
)
from sklearn.model_selection import GridSearchCV
import xgboost as xgb
import shap

# Configurar estilos
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("✅ Librerías importadas")
print(f"XGBoost versión: {xgb.__version__}")
print(f"SHAP versión: {shap.__version__}")

### 3.1 Cargar Datos Preprocesados

In [ ]:
# Cargar datos desde PASO 2
with open('../data/processed/preprocessed_data.pkl', 'rb') as f:
    data = pickle.load(f)

X_train = data['X_train']
X_val = data['X_val']
X_test = data['X_test']
y_train = data['y_train']
y_val = data['y_val']
y_test = data['y_test']
feature_names = data['feature_names']

print(f"✅ Datos cargados:")
print(f"  X_train: {X_train.shape}")
print(f"  X_val: {X_val.shape}")
print(f"  X_test: {X_test.shape}")
print(f"  Features: {len(feature_names)}")

### 3.2 Modelo Baseline: Regresión Logística

In [ ]:
print("\n" + "="*70)
print("ENTRENAMIENTO: REGRESIÓN LOGÍSTICA (BASELINE)")
print("="*70)

# Crear y entrenar el modelo
lr_model = LogisticRegression(max_iter=1000, random_state=42, n_jobs=-1)
lr_model.fit(X_train, y_train)

# Predicciones
y_train_pred_lr = lr_model.predict(X_train)
y_val_pred_lr = lr_model.predict(X_val)
y_test_pred_lr = lr_model.predict(X_test)

# Probabilidades para AUC
y_train_proba_lr = lr_model.predict_proba(X_train)[:, 1]
y_val_proba_lr = lr_model.predict_proba(X_val)[:, 1]
y_test_proba_lr = lr_model.predict_proba(X_test)[:, 1]

print("✅ Modelo entrenado")

In [ ]:
# Evaluar Baseline en TEST
lr_acc = accuracy_score(y_test, y_test_pred_lr)
lr_f1 = f1_score(y_test, y_test_pred_lr)
lr_auc = roc_auc_score(y_test, y_test_proba_lr)

print(f"\n📊 MÉTRICAS - TEST SET (Regresión Logística):")
print(f"  Accuracy: {lr_acc:.4f}")
print(f"  F1-Score: {lr_f1:.4f}")
print(f"  AUC-ROC:  {lr_auc:.4f}")

# Matriz de confusión
cm_lr = confusion_matrix(y_test, y_test_pred_lr)
print(f"\n📊 Matriz de Confusión:")
print(cm_lr)

# Reporte clasificación
print(f"\n📋 Classification Report:")
print(classification_report(y_test, y_test_pred_lr, target_names=['No Dropout', 'Dropout']))

### 3.3 Modelo Principal: XGBoost con GridSearchCV

In [ ]:
print("\n" + "="*70)
print("TUNING DE HIPERPARÁMETROS: XGBoost + GridSearchCV")
print("="*70)

# Definir parámetros a buscar (reducido para tiempo)
param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [4, 6, 8],
    'learning_rate': [0.01, 0.1],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0]
}

print(f"\n🔍 Grid Search Parameters:")
for param, values in param_grid.items():
    print(f"  {param}: {values}")

# Crear modelo base
xgb_base = xgb.XGBClassifier(
    objective='binary:logistic',
    random_state=42,
    n_jobs=-1,
    verbosity=0
)

# GridSearchCV
grid_search = GridSearchCV(
    estimator=xgb_base,
    param_grid=param_grid,
    cv=5,  # 5-fold cross-validation
    scoring='roc_auc',  # Métrica objetivo
    n_jobs=-1,
    verbose=1
)

print(f"\n⏳ Entrenando... (esto puede tomar ~2-3 minutos)")
grid_search.fit(X_train, y_train)
print(f"\n✅ GridSearchCV completado")

In [ ]:
# Mejores parámetros
best_params = grid_search.best_params_
print(f"\n🏆 MEJORES HIPERPARÁMETROS:")
for param, value in best_params.items():
    print(f"  {param}: {value}")

print(f"\n  Best CV Score (AUC-ROC): {grid_search.best_score_:.4f}")

# Obtener el mejor modelo
xgb_best = grid_search.best_estimator_
print(f"\n✅ Mejor modelo seleccionado")

In [ ]:
# Evaluar en todos los sets
print("\n" + "="*70)
print("EVALUACIÓN: XGBoost (Mejor Modelo)")
print("="*70)

# Predicciones
y_train_pred_xgb = xgb_best.predict(X_train)
y_val_pred_xgb = xgb_best.predict(X_val)
y_test_pred_xgb = xgb_best.predict(X_test)

# Probabilidades
y_train_proba_xgb = xgb_best.predict_proba(X_train)[:, 1]
y_val_proba_xgb = xgb_best.predict_proba(X_val)[:, 1]
y_test_proba_xgb = xgb_best.predict_proba(X_test)[:, 1]

# Métricas en TRAIN
xgb_train_acc = accuracy_score(y_train, y_train_pred_xgb)
xgb_train_f1 = f1_score(y_train, y_train_pred_xgb)
xgb_train_auc = roc_auc_score(y_train, y_train_proba_xgb)

print(f"\n📊 TRAIN SET:")
print(f"  Accuracy: {xgb_train_acc:.4f}")
print(f"  F1-Score: {xgb_train_f1:.4f}")
print(f"  AUC-ROC:  {xgb_train_auc:.4f}")

# Métricas en VALIDATION
xgb_val_acc = accuracy_score(y_val, y_val_pred_xgb)
xgb_val_f1 = f1_score(y_val, y_val_pred_xgb)
xgb_val_auc = roc_auc_score(y_val, y_val_proba_xgb)

print(f"\n📊 VALIDATION SET:")
print(f"  Accuracy: {xgb_val_acc:.4f}")
print(f"  F1-Score: {xgb_val_f1:.4f}")
print(f"  AUC-ROC:  {xgb_val_auc:.4f}")

# Métricas en TEST
xgb_acc = accuracy_score(y_test, y_test_pred_xgb)
xgb_f1 = f1_score(y_test, y_test_pred_xgb)
xgb_auc = roc_auc_score(y_test, y_test_proba_xgb)

print(f"\n📊 TEST SET (Principal):")
print(f"  Accuracy: {xgb_acc:.4f}")
print(f"  F1-Score: {xgb_f1:.4f}")
print(f"  AUC-ROC:  {xgb_auc:.4f}")

# Matriz de confusión
cm_xgb = confusion_matrix(y_test, y_test_pred_xgb)
print(f"\n📊 Matriz de Confusión (TEST):")
print(cm_xgb)

print(f"\n📋 Classification Report (TEST):")
print(classification_report(y_test, y_test_pred_xgb, target_names=['No Dropout', 'Dropout']))

### 3.4 Comparación de Modelos

In [ ]:
# Crear tabla comparativa
comparison_df = pd.DataFrame({
    'Modelo': ['Logistic Regression (Baseline)', 'XGBoost (Optimizado)'],
    'Accuracy': [lr_acc, xgb_acc],
    'F1-Score': [lr_f1, xgb_f1],
    'AUC-ROC': [lr_auc, xgb_auc]
})

print("\n" + "="*70)
print("COMPARACIÓN DE MODELOS (TEST SET)")
print("="*70)
print(comparison_df.to_string(index=False))

# Mejora
mejora_acc = (xgb_acc - lr_acc) / lr_acc * 100
mejora_f1 = (xgb_f1 - lr_f1) / lr_f1 * 100
mejora_auc = (xgb_auc - lr_auc) / lr_auc * 100

print(f"\n🚀 MEJORA (XGBoost vs Baseline):")
print(f"  Accuracy:  +{mejora_acc:.2f}%")
print(f"  F1-Score:  +{mejora_f1:.2f}%")
print(f"  AUC-ROC:   +{mejora_auc:.2f}%")

# Cumple objetivo?
objetivo = xgb_auc > 0.85
print(f"\n🎯 OBJETIVO (AUC-ROC > 0.85): {'✅ CUMPLIDO' if objetivo else '❌ NO CUMPLIDO'}")
print(f"   Valor obtenido: {xgb_auc:.4f}")

In [ ]:
# Guardar tabla de comparación
comparison_df.to_csv('../reports/03_model_comparison.csv', index=False)
print("✅ Tabla de comparación guardada: 03_model_comparison.csv")

### 3.5 Curvas ROC Comparativas

In [ ]:
# Calcular curvas ROC
fpr_lr, tpr_lr, _ = roc_curve(y_test, y_test_proba_lr)
fpr_xgb, tpr_xgb, _ = roc_curve(y_test, y_test_proba_xgb)

# Graficar
plt.figure(figsize=(10, 8))
plt.plot(fpr_lr, tpr_lr, label=f'Logistic Regression (AUC = {lr_auc:.4f})', linewidth=2.5, color='#e74c3c')
plt.plot(fpr_xgb, tpr_xgb, label=f'XGBoost (AUC = {xgb_auc:.4f})', linewidth=2.5, color='#2ecc71')
plt.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random Classifier')

plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate', fontsize=12)
plt.title('Curvas ROC - Comparación de Modelos', fontsize=14, fontweight='bold')
plt.legend(loc='lower right', fontsize=11)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('../figures/07_roc_curves_comparison.png', dpi=300, bbox_inches='tight')
plt.show()
print("✅ Figura guardada: 07_roc_curves_comparison.png")

### 3.6 Matrices de Confusión

In [ ]:
# Matrices de confusión lado a lado
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Baseline
sns.heatmap(cm_lr, annot=True, fmt='d', cmap='Reds', ax=axes[0], cbar=False,
            xticklabels=['No Dropout', 'Dropout'],
            yticklabels=['No Dropout', 'Dropout'])
axes[0].set_title(f'Logistic Regression\nAccuracy: {lr_acc:.4f}', fontweight='bold', fontsize=12)
axes[0].set_ylabel('Etiqueta Verdadera')
axes[0].set_xlabel('Predicción')

# XGBoost
sns.heatmap(cm_xgb, annot=True, fmt='d', cmap='Greens', ax=axes[1], cbar=False,
            xticklabels=['No Dropout', 'Dropout'],
            yticklabels=['No Dropout', 'Dropout'])
axes[1].set_title(f'XGBoost (Optimizado)\nAccuracy: {xgb_acc:.4f}', fontweight='bold', fontsize=12)
axes[1].set_ylabel('Etiqueta Verdadera')
axes[1].set_xlabel('Predicción')

plt.tight_layout()
plt.savefig('../figures/08_confusion_matrices.png', dpi=300, bbox_inches='tight')
plt.show()
print("✅ Figura guardada: 08_confusion_matrices.png")

### 3.7 Importancia de Features con SHAP

In [ ]:
print("\n" + "="*70)
print("EXPLICABILIDAD: SHAP Values")
print("="*70)

# Crear explicador SHAP
print("\n⏳ Calculando SHAP values... (esto toma ~1-2 minutos)")

# Usar TreeExplainer para XGBoost (más rápido)
explainer = shap.TreeExplainer(xgb_best)
shap_values = explainer.shap_values(X_test)

print("✅ SHAP values calculados")

In [ ]:
# Feature Importance (media de |SHAP values|)
feature_importance_shap = np.abs(shap_values).mean(axis=0)
feature_importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': feature_importance_shap
}).sort_values('Importance', ascending=False)

print(f"\n📊 Top 15 Features (SHAP Mean Absolute Value):")
print(feature_importance_df.head(15).to_string(index=False))

# Guardar
feature_importance_df.to_csv('../reports/03_shap_feature_importance.csv', index=False)
print("\n✅ Feature Importance guardado: 03_shap_feature_importance.csv")

In [ ]:
# Gráfico de Feature Importance
top_features_shap = feature_importance_df.head(15)

plt.figure(figsize=(10, 8))
plt.barh(range(len(top_features_shap)), top_features_shap['Importance'], color='steelblue')
plt.yticks(range(len(top_features_shap)), top_features_shap['Feature'])
plt.xlabel('SHAP Mean Absolute Value', fontsize=11, fontweight='bold')
plt.title('Top 15 Features más importantes (SHAP)', fontsize=13, fontweight='bold')
plt.gca().invert_yaxis()
plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig('../figures/09_shap_feature_importance.png', dpi=300, bbox_inches='tight')
plt.show()
print("✅ Figura guardada: 09_shap_feature_importance.png")

In [ ]:
# SHAP Summary Plot
plt.figure(figsize=(10, 8))
shap.summary_plot(shap_values, X_test, feature_names=feature_names, plot_type='bar', show=False)
plt.tight_layout()
plt.savefig('../figures/10_shap_summary_bar.png', dpi=300, bbox_inches='tight')
plt.show()
print("✅ Figura guardada: 10_shap_summary_bar.png")

### 3.8 XGBoost Feature Importance (alternativo)

In [ ]:
# Feature importance del modelo XGBoost
xgb_importance = xgb_best.feature_importances_
xgb_importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': xgb_importance
}).sort_values('Importance', ascending=False)

print(f"\n📊 Top 15 Features (XGBoost Gain):")
print(xgb_importance_df.head(15).to_string(index=False))

# Gráfico
top_xgb = xgb_importance_df.head(15)
plt.figure(figsize=(10, 8))
plt.barh(range(len(top_xgb)), top_xgb['Importance'], color='seagreen')
plt.yticks(range(len(top_xgb)), top_xgb['Feature'])
plt.xlabel('Gain (Importancia)', fontsize=11, fontweight='bold')
plt.title('Top 15 Features - XGBoost Feature Importance', fontsize=13, fontweight='bold')
plt.gca().invert_yaxis()
plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig('../figures/11_xgboost_feature_importance.png', dpi=300, bbox_inches='tight')
plt.show()
print("✅ Figura guardada: 11_xgboost_feature_importance.png")

### 3.9 Guardar el Mejor Modelo

In [ ]:
# Guardar el mejor modelo XGBoost
import joblib

model_path = '../models/best_xgboost.pkl'
joblib.dump(xgb_best, model_path)

print(f"✅ Modelo guardado: {model_path}")

# También guardar GridSearch
grid_path = '../models/grid_search_results.pkl'
joblib.dump(grid_search, grid_path)
print(f"✅ GridSearch guardado: {grid_path}")

### 3.10 Resumen del Modelado

In [ ]:
resumen_modelado = f"""
═══════════════════════════════════════════════════════════════════════════════
RESUMEN DEL MODELADO ML
═══════════════════════════════════════════════════════════════════════════════

📊 DATOS DE ENTRENAMIENTO:
  • Train: {X_train.shape[0]:,} muestras
  • Validation: {X_val.shape[0]:,} muestras
  • Test: {X_test.shape[0]:,} muestras
  • Features: {X_train.shape[1]}

🏆 MODELO 1: REGRESIÓN LOGÍSTICA (BASELINE)
  • Algoritmo: Logistic Regression (max_iter=1000)
  • Objective: Binary Classification
  
  Métricas (TEST SET):
    - Accuracy:  {lr_acc:.4f}
    - F1-Score:  {lr_f1:.4f}
    - AUC-ROC:   {lr_auc:.4f}

🎯 MODELO 2: XGBoost OPTIMIZADO (PRINCIPAL)
  • Algoritmo: XGBoost Classifier
  • Tuning: GridSearchCV (5-fold CV, scoring='roc_auc')
  
  Mejores Hiperparámetros:
"""

for param, value in best_params.items():
    resumen_modelado += f"    - {param}: {value}\n"

resumen_modelado += f"""
  Métricas por Dataset:
    TRAIN SET:
      - Accuracy:  {xgb_train_acc:.4f}
      - F1-Score:  {xgb_train_f1:.4f}
      - AUC-ROC:   {xgb_train_auc:.4f}
    
    VALIDATION SET:
      - Accuracy:  {xgb_val_acc:.4f}
      - F1-Score:  {xgb_val_f1:.4f}
      - AUC-ROC:   {xgb_val_auc:.4f}
    
    TEST SET:
      - Accuracy:  {xgb_acc:.4f}
      - F1-Score:  {xgb_f1:.4f}
      - AUC-ROC:   {xgb_auc:.4f}

📈 COMPARACIÓN (XGBoost vs Baseline en TEST):
  • Accuracy:  {xgb_acc:.4f} vs {lr_acc:.4f} → Mejora: +{mejora_acc:.2f}%
  • F1-Score:  {xgb_f1:.4f} vs {lr_f1:.4f} → Mejora: +{mejora_f1:.2f}%
  • AUC-ROC:   {xgb_auc:.4f} vs {lr_auc:.4f} → Mejora: +{mejora_auc:.2f}%

🎯 OBJETIVO INVESTIGACIÓN:
  Pregunta: ¿AUC-ROC > 0.85?
  Respuesta: {'✅ SÍ - CUMPLIDO' if objetivo else '❌ NO - No cumplido'}
  Valor obtenido: {xgb_auc:.4f}

🔍 EXPLICABILIDAD (SHAP):
  • Valores SHAP calculados para {X_test.shape[0]:,} muestras de test
  • Top 5 Features más importantes:
"""

for idx, row in feature_importance_df.head(5).iterrows():
    resumen_modelado += f"    {idx+1}. {row['Feature']}: {row['Importance']:.4f}\n"

resumen_modelado += f"""
📦 ARTEFACTOS GUARDADOS:
  ✓ models/best_xgboost.pkl (mejor modelo)
  ✓ models/grid_search_results.pkl (historial búsqueda)
  ✓ reports/03_model_comparison.csv
  ✓ reports/03_shap_feature_importance.csv
  ✓ figures/07_roc_curves_comparison.png
  ✓ figures/08_confusion_matrices.png
  ✓ figures/09_shap_feature_importance.png
  ✓ figures/10_shap_summary_bar.png
  ✓ figures/11_xgboost_feature_importance.png

✅ MODELADO COMPLETADO
═══════════════════════════════════════════════════════════════════════════════
"""

print(resumen_modelado)

# Guardar resumen
with open('../reports/03_modeling_summary.txt', 'w', encoding='utf-8') as f:
    f.write(resumen_modelado)
print("\n✅ Resumen guardado: 03_modeling_summary.txt")

In [ ]:
print("\n" + "="*70)
print("✅ PASO 3 COMPLETADO - Listo para PASO 4 (LLM Explicabilidad)")
print("="*70)